# C3 lab: an index you can trust (ids, hashes, cache, manifest, rebuild rule)

Lesson: [`concepts/C3_embedding_indexing.md`](../concepts/C3_embedding_indexing.md), Part H.1. Concept card: [02 C3](../02_RAG_MASTER_LEARNING_PLAN.md#c3).

**How this lab works (blank-file first).**
- Each task has a **task cell** (what to build), an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. You may look up the Python, NumPy and sentence-transformers docs. The reference script `sources/verification/C3/c3_measure.py` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- Where a task says **Predict first**, fill the prediction cell **before** you run anything. The CHECK cells refuse to run without it.
- Some CHECK cells compare **fingerprints** (`_fp`), so the expected answers stay hidden until you have predicted.
- **No paid API calls.** Everything runs on the local MiniLM models from your Hugging Face cache. Task 8 (Cohere) is optional and **off** by default (`RUN_COHERE = False`).
- Files: the lab writes only to `rag_learning/index/c3_lab/` and `rag_learning/.cache/emb_c3_lab/`. Both folders are git-ignored and are emptied when you run the setup cell.
- Kernel: the project uv environment (`rag-learning`). Run the cells top to bottom.

## Setup (given: see C1 and C2)

Run this cell. It gives you, ready-made, the parts that earlier lessons own:
- the climate PDF's pages, `full` (pages joined with one `"\n"`) and its **snapshot id**, computed with the exact 06 §5.3 recipe (**given, see C1**);
- `CFG`, the v0 chunking parameters (06 §2.4 `chunking:` block), and `make_chunks(doc_id, snapshot_id, pages, cfg, params_hash_fn)`: the v0 per-page `recursive_char` chunker (1,000/200 characters) that returns **06 §2.3 chunk records** as dicts (**given, see C2**: it gives the same 97 records as the C2 lab). It needs *your* `params_hash` function, which is Task 1;
- the two local models (`m6` = all-MiniLM-L6-v2, `m3` = paraphrase-MiniLM-L3-v2), their cache specs, and `Counting(model)`, an encoder that counts how many texts it really sends to the model;
- `_fp` (a fingerprint) and `_names` (the names a function uses), for the CHECK cells.

In [ ]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # never download; both models are in the local cache
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import bisect, hashlib, json, pathlib, re, shutil
import numpy as np
import pymupdf
from sentence_transformers import SentenceTransformer

PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"
Q_D002 = "In what year was the Paris Agreement adopted?"     # 06 §6.7 seed D-002 (draft item; demo only)

_cwd = pathlib.Path.cwd()
WORK = pathlib.Path(os.environ.get("C3_WORK_DIR") or (_cwd.parent if _cwd.name == "labs" else _cwd))
assert os.environ.get("C3_WORK_DIR") or (WORK / "pyproject.toml").exists(), f"open the notebook from rag_learning/labs (WORK={WORK})"
INDEX_ROOT = WORK / "index" / "c3_lab"          # git-ignored
LAB_CACHE = WORK / ".cache" / "emb_c3_lab"      # git-ignored
shutil.rmtree(INDEX_ROOT, ignore_errors=True)
shutil.rmtree(LAB_CACHE, ignore_errors=True)


def sha256_hex(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def snapshot_id_of(pages, doc_id="climate"):
    """06 §5.3 recipe, exactly (given, see C1)."""
    doc_text_sha256 = {doc_id: sha256_hex("\n".join(pages))}
    backend, version, options, cleaning_version = "pymupdf", pymupdf.VersionBind, {}, "none"
    parts = ["snapshot-v1", f"parser={backend}@{version}", "options=" + json.dumps(options, sort_keys=True),
             f"cleaning={cleaning_version}", "page_join=\\n"]
    parts += [f"{d}:{doc_text_sha256[d]}" for d in sorted(doc_text_sha256)]
    return hashlib.sha256("|".join(parts).encode("utf-8")).hexdigest()[:12]


PDF_SHA256 = hashlib.sha256(pathlib.Path(PDF).read_bytes()).hexdigest()
pages = [p.get_text() for p in pymupdf.open(PDF)]
full = "\n".join(pages)
SNAP = snapshot_id_of(pages)
assert SNAP == "ea07572562ac", "this is not the frozen Corpus D text (pymupdf must be 1.26.7)"


def edited_pages():
    """A copy of the pages with 9 characters inserted on page 10 (a 'document update')."""
    e = list(pages)
    e[9] = pages[9][:100] + " (edited)" + pages[9][100:]
    return e


# ---------------- given, see C2: the v0 recursive_char chunker (per page) -> 06 §2.3 chunk records
def _split_spans(text, start, end, seps, size):
    if end - start <= size:
        return [(start, end)]
    if not seps or seps[0] == "":                 # "" = last resort: single characters, merged below
        return [(s, s + 1) for s in range(start, end)]
    sep, rest, pieces, i = seps[0], seps[1:], [], start
    while i < end:
        j = text.find(sep, i, end)
        if j == -1:
            pieces.append((i, end))
            break
        pieces.append((i, j + len(sep)))
        i = j + len(sep)
    out = []
    for a, b in pieces:
        out.extend([(a, b)] if b - a <= size else _split_spans(text, a, b, rest, size))
    return out


def recursive_spans(text, size, overlap, seps):
    if not text:
        return []
    spans, cur = [], []
    for p in _split_spans(text, 0, len(text), seps, size):
        if cur and p[1] - cur[0][0] > size:
            spans.append((cur[0][0], cur[-1][1]))
            keep = []
            for q in reversed(cur):
                if cur[-1][1] - q[0] <= overlap:
                    keep.insert(0, q)
                else:
                    break
            cur = keep
            while cur and p[1] - cur[0][0] > size:
                cur.pop(0)
        cur.append(p)
    if cur:
        spans.append((cur[0][0], cur[-1][1]))
    return spans


def make_chunks(doc_id, snapshot_id, pages, cfg, params_hash_fn):
    full_, starts, pos = "\n".join(pages), [], 0
    for p in pages:
        starts.append(pos)
        pos += len(p) + 1
    size, ov, seps = cfg["chunk_size"], cfg["chunk_overlap"], cfg["separators"]
    if cfg["scope"] == "per_page":
        spans = [(starts[i] + a, starts[i] + b) for i, p in enumerate(pages) for a, b in recursive_spans(p, size, ov, seps)]
    else:
        spans = recursive_spans(full_, size, ov, seps)
    h = params_hash_fn(cfg)
    out = []
    for seq, (a, b) in enumerate(spans):
        segs = []
        for pg in range(bisect.bisect_right(starts, a), bisect.bisect_right(starts, b - 1) + 1):
            lo, hi = starts[pg - 1], (starts[pg] if pg < len(starts) else len(full_))
            segs.append({"page": pg, "char_start": max(a, lo) - lo, "char_end": min(b, hi) - lo})
        out.append({"doc_id": doc_id, "snapshot_id": snapshot_id, "chunk_id": f"{doc_id}:{snapshot_id}:{h}:{seq:05d}",
                    "seq": seq, "char_start": a, "char_end": b, "page_segments": segs, "text": full_[a:b],
                    "text_sha256": sha256_hex(full_[a:b]), "chunker": cfg["name"], "params_hash": h, "meta": {}})
    return out


CFG = {"name": "recursive_char", "unit": "chars", "chunk_size": 1000, "chunk_overlap": 200,
       "separators": ["\n\n", "\n", " ", ""], "scope": "per_page"}     # 06 §2.4 `chunking:` block = the C2 lab's PARAMS
C2_JSONL_SHA256 = "d4108b8936093394c6eb50d4c22e8d4013d9a048fb38d5769281af4aadb3011b"   # the v0 chunks.jsonl of the C2 lab

# ---------------- the two local embedders and their cache specs
L6 = "sentence-transformers/all-MiniLM-L6-v2"
L3 = "sentence-transformers/paraphrase-MiniLM-L3-v2"
SPEC_L6 = dict(provider="sentence_transformers", model_id=L6, revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
               input_type="none", preproc_version="none")
SPEC_L3 = dict(provider="sentence_transformers", model_id=L3, revision="4ca70771034acceecb2e72475f72050fcdde4ddc",
               input_type="none", preproc_version="none")
m6 = SentenceTransformer(L6, device="cpu")
m3 = SentenceTransformer(L3, device="cpu")


class Counting:
    """encode(list_of_texts) -> float32 array of RAW model vectors; counts texts really sent to the model."""
    def __init__(self, model):
        self.model, self.n_texts, self.n_calls = model, 0, 0

    def __call__(self, texts):
        self.n_texts += len(texts)
        self.n_calls += 1
        return self.model.encode(list(texts), normalize_embeddings=False, convert_to_numpy=True).astype(np.float32)


def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]


def _names(fn):
    """Every global or attribute name a function's code uses (also inside nested code)."""
    out, todo = set(), [fn.__code__]
    while todo:
        c = todo.pop()
        out |= set(c.co_names)
        todo += [k for k in c.co_consts if hasattr(k, "co_names")]
    return out


print(len(pages), "pages,", len(full), "characters; snapshot_id", SNAP, "; work dir", WORK)

## Task 1: `params_hash`, the name of a chunking configuration

You wrote a `params_hash` in the C2 lab (Task 7). Write it **again from memory**, without opening C2 (retrieval practice). This CHECK tests the properties an index relies on.

Write `params_hash(cfg: dict) -> str`:
- the **same config must always give the same hash**, in any process, on any day, whatever the order of the keys;
- **any change to any field** must give a different hash;
- the course recipe (the same as the C2 lab, so that your ids match C2's records and C4's traces): the SHA-256 of `json.dumps(cfg, sort_keys=True)` (default separators) encoded as UTF-8, hex, **first 12 characters**. `cfg` is the 06 §2.4 `chunking:` block, `CFG`.

Why: the hash goes into every chunk id (`doc_id:snapshot_id:params_hash:seq`) and into the index folder name. If two different configs could share a hash, two different chunks could share an id.

**Predict first.** Change only `scope` from `"per_page"` to `"cross_page"`. Of the 97 per-page chunk ids, how many will also appear among the cross-page chunk ids? Write a number.

In [ ]:
my_prediction_ids_kept = None   # replace None with a number, BEFORE you write params_hash

In [ ]:
# Your code here (Task 1). Blank-file first: write it yourself; no copying from c3_measure.py.

In [ ]:
# CHECK Task 1 (do not edit)
assert my_prediction_ids_kept is not None, "write your prediction first (it is part of the evidence)"
assert "hash" not in _names(params_hash), "do not use Python's built-in hash(): it changes between processes (PYTHONHASHSEED)"
_h = params_hash(CFG)
assert isinstance(_h, str) and re.fullmatch(r"[0-9a-f]{12}", _h), f"return 12 lowercase hex characters, got {_h!r}"
assert params_hash(dict(reversed(list(CFG.items())))) == _h, "the same config with its keys in another order must give the same hash (sort_keys)"
assert params_hash(json.loads(json.dumps(CFG))) == _h, "a copy of the config must give the same hash"
for _k, _v in [("name", "recursive"), ("chunk_size", 1001), ("chunk_overlap", 100), ("unit", "tokens"), ("scope", "cross_page"),
               ("separators", ["\n\n", "\n", " "])]:
    assert params_hash({**CFG, _k: _v}) != _h, f"changing {_k!r} must change the hash: your hash ignores it"
assert _fp(_h) == "05434e71a00f90df", "deterministic and sensitive, but not the course recipe yet: sha256 of json.dumps(cfg, sort_keys=True), first 12 hex"
_a = make_chunks("climate", SNAP, pages, CFG, params_hash)
_b = make_chunks("climate", SNAP, pages, dict(CFG), params_hash)
assert len(_a) == 97 and [c["chunk_id"] for c in _a] == [c["chunk_id"] for c in _b], "same inputs must give the same chunk ids"
assert all(re.fullmatch(r"climate:ea07572562ac:[0-9a-f]{12}:\d{5}", c["chunk_id"]) for c in _a)
_kept = len({c["chunk_id"] for c in _a} & {c["chunk_id"] for c in make_chunks("climate", SNAP, pages, {**CFG, "scope": "cross_page"}, params_hash)})
print(f"Task 1 OK: params_hash = {_h}; first id {_a[0]['chunk_id']}. Ids kept after the scope change: {_kept} (you predicted {my_prediction_ids_kept})")

Given: the chunk records for the rest of the lab.

In [ ]:
chunks = make_chunks("climate", SNAP, pages, CFG, params_hash)     # given (see C2), named by YOUR params_hash
texts = [c["text"] for c in chunks]
print(len(chunks), "chunks; first record without its text:", {k: v for k, v in chunks[0].items() if k != "text"})
_jl = "".join(json.dumps(c, ensure_ascii=False) + "\n" for c in chunks).encode("utf-8")
print("same 97 records as the C2 lab's v0 chunks.jsonl:", hashlib.sha256(_jl).hexdigest() == C2_JSONL_SHA256)
_c2 = WORK / "labs" / "C2_out" / f"{SNAP}_{params_hash(CFG)}" / "chunks.jsonl"
if _c2.exists():
    print("your own C2 output", _c2.parent.name, "matches:", hashlib.sha256(_c2.read_bytes()).hexdigest() == C2_JSONL_SHA256)

## Task 2: the embedding cache key

Write `embedding_cache_key(text, *, provider, model_id, revision, input_type, preproc_version) -> str`:
- 06 §2.4: the SHA-256 hex (all 64 characters) of `provider|model_id|revision|input_type|preproc_version|text`, joined with `"|"`, encoded as UTF-8;
- `revision` may be `None` for an API model: use `str(revision)`;
- if any of the five **meta** fields contains `"|"`, raise `ValueError` (the text is last, so the text itself may contain `"|"`).

Why: a cached vector may only be reused when **everything that shaped it** is the same. The chunk id is **not** that: it changes when the text has not, and it stays the same when the model changes.

**Predict first.** With Cohere, a user's question happens to be exactly the same text as one of your chunks. Should the question reuse the chunk's cached vector? Write `"yes"` or `"no"`.

In [ ]:
my_prediction_same_vector = None   # "yes" or "no", BEFORE you code

In [ ]:
# Your code here (Task 2). Blank-file first: write it yourself; no copying from c3_measure.py.

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_same_vector in ("yes", "no"), "write your prediction first"
_k0 = embedding_cache_key("hello|world", **SPEC_L6)
assert isinstance(_k0, str) and re.fullmatch(r"[0-9a-f]{64}", _k0), "return the full 64-character sha256 hex"
for _f, _v in [("provider", "cohere"), ("model_id", L3), ("revision", "other"), ("input_type", "search_query"),
               ("preproc_version", "tabs_to_spaces")]:
    assert embedding_cache_key("hello|world", **{**SPEC_L6, _f: _v}) != _k0, f"changing {_f!r} must change the key"
assert embedding_cache_key("hello|worle", **SPEC_L6) != _k0, "the text itself must be part of the key"
try:
    embedding_cache_key("x", **{**SPEC_L6, "model_id": "a|b"})
    raise AssertionError("a '|' inside a meta field must raise ValueError (else two different keys can collide)")
except ValueError:
    pass
assert embedding_cache_key("x", **{**SPEC_L6, "revision": None}) == embedding_cache_key("x", **{**SPEC_L6, "revision": "None"}), "use str(revision)"
assert _fp(_k0) == "14b09a432ef59943", "every field is in the key, but not in the 06 order or format yet: provider|model_id|revision|input_type|preproc_version|text"
print("Task 2 OK:", _k0[:16] + "…", "| your prediction:", my_prediction_same_vector)

## Task 3: a content-keyed embedding cache

Write `cached_embed(texts, spec, encode, cache_dir) -> (V, stats)`:
- `spec` is one of the given specs (`SPEC_L6`, `SPEC_L3`): the keyword arguments of `embedding_cache_key`;
- for each text, look for `cache_dir / f"{key}.npy"`. Send **only the missing texts** to `encode` (one call with the list), then save each new **raw** vector as `<key>.npy` (`np.save`);
- return `V`: the vectors in the order of `texts`, **L2-normalised**, `float32`, shape `(len(texts), dim)`; and `stats = {"hits": ..., "misses": ...}` counted over `texts`;
- load with `np.load(..., allow_pickle=False)`.

**Predict first.** Page 10 is edited: 9 characters are inserted near its start (`edited_pages()`). The page-10 text changes, so the snapshot id changes. Of the 97 chunks: how many **chunk ids** change, and how many **cache misses** will the rebuild have? Write two numbers.

In [ ]:
my_prediction_ids_changed = None   # a number, BEFORE you code
my_prediction_misses = None        # a number, BEFORE you code

In [ ]:
# Your code here (Task 3). Blank-file first: write it yourself; no copying from c3_measure.py.

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_ids_changed is not None and my_prediction_misses is not None, "write both predictions first"
_c = LAB_CACHE / "check3"
shutil.rmtree(_c, ignore_errors=True)
_e = Counting(m6)
_V, _s = cached_embed(texts, SPEC_L6, _e, _c)
assert isinstance(_V, np.ndarray) and _V.shape == (97, 384) and _V.dtype == np.float32, f"V must be float32 (97, 384), got {getattr(_V, 'shape', None)} {getattr(_V, 'dtype', None)}"
assert _s == {"hits": 0, "misses": 97} and _e.n_texts == 97, f"empty cache: expected 97 misses and 97 texts sent, got {_s}, sent {_e.n_texts}"
_e2 = Counting(m6)
_V2, _s2 = cached_embed(texts, SPEC_L6, _e2, _c)
assert _s2 == {"hits": 97, "misses": 0} and _e2.n_texts == 0, f"second build: every text must come from the cache, got {_s2}, sent {_e2.n_texts}"
assert np.array_equal(_V, _V2), "vectors from the cache must equal the first build"
_pe = edited_pages()
_ce = make_chunks("climate", snapshot_id_of(_pe), _pe, CFG, params_hash)
_e3 = Counting(m6)
_, _s3 = cached_embed([c["text"] for c in _ce], SPEC_L6, _e3, _c)
_changed = len({c["chunk_id"] for c in _ce} - {c["chunk_id"] for c in chunks})
assert _fp([_s3["misses"], _e3.n_texts, _changed]) == "b4840455515c2cac", f"after the page-10 edit: misses {_s3['misses']}, texts sent {_e3.n_texts}: only changed TEXTS may be re-embedded"
_e4 = Counting(m3)
_V3, _s4 = cached_embed(texts, SPEC_L3, _e4, _c)
assert _e4.n_texts == 97, f"switching to another model must re-embed every text, but only {_e4.n_texts} were sent: is the model in your key?"
assert np.allclose(np.linalg.norm(_V3, axis=1), 1.0, atol=1e-5), "rows of V must have length 1 (L3 does not normalise for you)"
print(f"Task 3 OK. Edit: {_changed} ids changed (you predicted {my_prediction_ids_changed}), {_s3['misses']} misses (you predicted {my_prediction_misses}).")

Given: embed the 97 chunks for the index (cold cache, then reused in Task 4).

In [ ]:
V, emb_stats = cached_embed(texts, SPEC_L6, Counting(m6), LAB_CACHE)
print(V.shape, V.dtype, emb_stats)

## Task 4: write the three index files

**4a.** Build `MANIFEST`, a dict with exactly these keys (the v0 interface; C4 reads it):

| Key | Value |
|---|---|
| `snapshot_id` | `SNAP` |
| `docs` | `[{"doc_id": "climate", "pdf_sha256": PDF_SHA256, "doc_text_sha256": <sha256 of full>}]` |
| `parser` | `{"backend": "pymupdf", "version": pymupdf.VersionBind}` |
| `chunker` | `CFG` (exactly the dict that `params_hash` hashes) |
| `params_hash` | `params_hash(CFG)` |
| `embedding` | `{"provider", "model_id", "revision"` (from `SPEC_L6`), `"input_type": {"documents": "none", "queries": "none"}, "preproc_version": "none", "normalize": True, "dim": 384, "max_seq_length": m6.max_seq_length}` |
| `embed_hash` | `embed_hash(MANIFEST["embedding"])` (4a below) |
| `library_versions` | a dict, e.g. numpy, sentence-transformers and torch versions |
| `created_utc` | a string |

Also write `embed_hash(embedding) -> str`: the same recipe as `params_hash`, over the embedding block **without** the measured fields `dim` and `max_seq_length`, i.e. over `provider, model_id, revision, input_type, normalize, preproc_version` (course interface, 2026-10-08).

**4b.** Write `write_index(index_dir, chunks, V, manifest) -> dict`:
- `vectors.npy`: `V` as float32 (`np.save`); row *i* belongs to line *i* of `chunks.jsonl`;
- `chunks.jsonl`: one `json.dumps(chunk, ensure_ascii=False)` per line, UTF-8, `"\n"` line ends, in the same order;
- `manifest.json`: the manifest **plus** `"files": {"vectors.npy": {"sha256", "shape", "dtype"}, "chunks.jsonl": {"sha256", "lines"}}` (sha256 of each file's bytes); return that dict;
- **never overwrite a different index**: if `index_dir/manifest.json` exists and describes another snapshot, chunker or embedding, raise an exception and touch nothing.

Then run it: `written = write_index(INDEX_DIR, chunks, V, MANIFEST)` with `INDEX_DIR = INDEX_ROOT / SNAP / params_hash(CFG) / embed_hash(MANIFEST["embedding"])` (the course layout `index/<snapshot_id>/<params_hash>/<embed_hash>/`).

**Predict first.** How many bytes will `vectors.npy` be? (A `.npy` file has a 128-byte header here.)

In [ ]:
my_prediction_npy_bytes = None   # a number, BEFORE you code

In [ ]:
# Your code here (Task 4): MANIFEST, write_index, then written = write_index(INDEX_DIR, chunks, V, MANIFEST)

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_npy_bytes is not None, "write your prediction first"
_eh = embed_hash(MANIFEST["embedding"])
assert isinstance(_eh, str) and re.fullmatch(r"[0-9a-f]{12}", _eh), "embed_hash: 12 lowercase hex characters"
assert embed_hash({**MANIFEST["embedding"], "dim": 999, "max_seq_length": 1}) == _eh, "embed_hash must ignore the measured fields dim and max_seq_length"
for _k, _v in [("provider", "cohere"), ("model_id", L3), ("revision", "x"), ("input_type", {"documents": "search_document", "queries": "none"}),
               ("normalize", False), ("preproc_version", "tabs_to_spaces")]:
    assert embed_hash({**MANIFEST["embedding"], _k: _v}) != _eh, f"changing {_k!r} must change embed_hash"
assert _fp(_eh) == "c2251aec9abca0a9", "embed_hash: sha256 of json.dumps(<the six fields>, sort_keys=True), first 12 hex"
assert INDEX_DIR == INDEX_ROOT / SNAP / params_hash(CFG) / _eh, "use INDEX_DIR = INDEX_ROOT / SNAP / params_hash(CFG) / embed_hash(MANIFEST['embedding'])"
assert sorted(p.name for p in INDEX_DIR.iterdir()) == ["chunks.jsonl", "manifest.json", "vectors.npy"], "exactly three files"
_m = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))
assert set(_m) == {"snapshot_id", "docs", "parser", "chunker", "params_hash", "embedding", "embed_hash", "library_versions", "created_utc", "files"}, f"manifest keys: {sorted(_m)}"
assert _m["embed_hash"] == _eh, "record embed_hash in the manifest"
assert _m["snapshot_id"] == SNAP and _m["chunker"] == CFG and _m["params_hash"] == params_hash(_m["chunker"])
assert _m["docs"] == [{"doc_id": "climate", "pdf_sha256": PDF_SHA256, "doc_text_sha256": sha256_hex(full)}], "docs entry"
_emb = _m["embedding"]
assert {k: _emb[k] for k in ("provider", "model_id", "revision")} == {k: SPEC_L6[k] for k in ("provider", "model_id", "revision")}
assert _emb["input_type"] == {"documents": "none", "queries": "none"} and _emb["preproc_version"] == "none" and _emb["normalize"] is True
assert _emb["dim"] == 384 and _emb["max_seq_length"] == 256
_Vd = np.load(INDEX_DIR / "vectors.npy", allow_pickle=False)
assert _Vd.dtype == np.float32 and _Vd.shape == (97, 384) and np.allclose(np.linalg.norm(_Vd, axis=1), 1, atol=1e-5)
_lines = (INDEX_DIR / "chunks.jsonl").read_bytes().decode("utf-8").split("\n")
assert _lines[-1] == "" and len(_lines) == 98 and "\r" not in "".join(_lines), "97 lines, each ending in \\n"
_cl = [json.loads(x) for x in _lines[:-1]]
assert _cl == chunks, "line i must be chunk i, all 06 fields kept"
assert all(full[c["char_start"]:c["char_end"]] == c["text"] for c in _cl)
for _n in ("vectors.npy", "chunks.jsonl"):
    assert _m["files"][_n]["sha256"] == hashlib.sha256((INDEX_DIR / _n).read_bytes()).hexdigest(), f"files.{_n}.sha256 must hash the file bytes"
assert _m["files"]["vectors.npy"]["shape"] == [97, 384] and _m["files"]["vectors.npy"]["dtype"] == "float32" and _m["files"]["chunks.jsonl"]["lines"] == 97
assert written == _m, "return the manifest you wrote (with files)"
_before = {p.name: p.read_bytes() for p in INDEX_DIR.iterdir()}
try:
    _e3 = {**MANIFEST["embedding"], "model_id": L3, "revision": SPEC_L3["revision"]}
    write_index(INDEX_DIR, chunks, V, {**MANIFEST, "embedding": _e3, "embed_hash": embed_hash(_e3)})
    _over = True
except Exception:
    _over = False
assert not _over, "writing a DIFFERENT index into an existing folder must raise (never overwrite in place)"
assert {p.name: p.read_bytes() for p in INDEX_DIR.iterdir()} == _before, "the refused write must not change any file"
_size = (INDEX_DIR / "vectors.npy").stat().st_size
print(f"Task 4 OK: vectors.npy {_size} bytes (you predicted {my_prediction_npy_bytes}); chunks.jsonl {(INDEX_DIR / 'chunks.jsonl').stat().st_size} bytes")

## Task 5: the rebuild rule

Given below: `RUN`, the run config of v0 (the parts that matter here). Write `rebuild_reasons(manifest, run_cfg) -> list`:
- return `[]` when this index can be reused for that run config;
- otherwise a list of the reasons, from `"snapshot"`, `"chunker"`, `"embedding"`, **in that order**;
- `"chunker"` when `params_hash(run_cfg["chunker"])` differs from the manifest's; `"snapshot"` when the snapshot ids differ;
- `"embedding"` when anything that shaped the **stored document vectors** differs: provider, model id, revision, the **documents** input type, preprocessing version, normalisation. Decide for yourself which other settings do **not** need a rebuild.

**Predict first.** For each change, write `"reuse"` or `"rebuild"` in the dict below.

In [ ]:
RUN = {"snapshot_id": SNAP, "chunker": CFG, "embedding": MANIFEST["embedding"], "retrieval": {"k": 4},
       "generation": {"provider": "groq", "model_id": "${GROQ_MODEL_ID}", "temperature": 0.0},
       "rerank": None, "prompt_version": "answer_v1"}           # given

my_predictions_rebuild = {          # "reuse" or "rebuild", BEFORE you code
    "k 4 -> 8": None,
    "Groq model changed": None,
    "chunk_overlap 200 -> 100": None,
    "query input_type none -> search_query": None,
    "normalize True -> False": None,
    "one page of the PDF edited": None,
}

In [ ]:
# Your code here (Task 5). Blank-file first: write it yourself; no copying from c3_measure.py.

In [ ]:
# CHECK Task 5 (do not edit)
assert all(v in ("reuse", "rebuild") for v in my_predictions_rebuild.values()), "fill every prediction first"
_E = MANIFEST["embedding"]
_SE = snapshot_id_of(edited_pages())
_scen = {
    "nothing changed": {}, "k 4 -> 8": {"retrieval": {"k": 8}},
    "LLM model changed": {"generation": {"provider": "groq", "model_id": "other", "temperature": 0.0}},
    "prompt answer_v1 -> v2": {"prompt_version": "answer_v2"}, "reranker added": {"rerank": {"name": "cross_encoder"}},
    "query input_type changed": {"embedding": {**_E, "input_type": {"documents": "none", "queries": "search_query"}}},
    "chunk size 1000 -> 2000": {"chunker": {**CFG, "chunk_size": 2000}}, "overlap 200 -> 100": {"chunker": {**CFG, "chunk_overlap": 100}},
    "scope per_page -> cross_page": {"chunker": {**CFG, "scope": "cross_page"}},
    "PDF text changed (new snapshot)": {"snapshot_id": _SE},
    "embedding model L6 -> L3": {"embedding": {**_E, "model_id": L3, "revision": SPEC_L3["revision"]}},
    "same model, new revision": {"embedding": {**_E, "revision": "0000000"}},
    "document input_type changed": {"embedding": {**_E, "input_type": {"documents": "search_document", "queries": "none"}}},
    "preprocessing (tabs->spaces) added": {"embedding": {**_E, "preproc_version": "tabs_to_spaces"}},
    "normalize true -> false": {"embedding": {**_E, "normalize": False}},
    "new snapshot AND new model": {"snapshot_id": _SE, "embedding": {**_E, "model_id": L3, "revision": SPEC_L3["revision"]}},
}
_got = {k: rebuild_reasons(written, {**RUN, **v}) for k, v in _scen.items()}
assert all(isinstance(v, list) for v in _got.values()), "return a list"
assert _got["nothing changed"] == [], "an unchanged config must reuse the index"
assert _got["k 4 -> 8"] == [] and _got["LLM model changed"] == [] and _got["reranker added"] == [], "query-time settings never need a rebuild"
for _k, _v in _got.items():
    print(f"  {_k:36s} -> {_v if _v else 'reuse'}")
assert _fp([_got[k] for k in _scen]) == "d4de412a44e2aff8", "not all 16 decisions are right yet (or the reasons are not in the order snapshot, chunker, embedding)"
print("Task 5 OK. Compare with your predictions:", my_predictions_rebuild)

## Task 6: load the index, and fail closed

Write `load_index(index_dir, run_cfg) -> (V, chunks, manifest)`. Raise `ValueError` (or a subclass) listing **every** problem found, if any:
- a file's sha256 differs from `manifest["files"]`;
- `vectors.npy` is not float32 of the recorded shape, or its rows are not unit length (tolerance 1e-5);
- the number of vectors differs from the number of chunk lines;
- a chunk line whose `snapshot_id` or `params_hash` differs from the manifest, or whose `text_sha256` does not match its text;
- the manifest's own `params_hash` or `embed_hash` does not match its `chunker` or `embedding` block (recompute both);
- `rebuild_reasons(manifest, run_cfg)` is not empty.

Rows and lines are matched by **position** (row *i* ↔ line *i*); do not assume `seq == i`: `seq` restarts in every document.

Load `vectors.npy` with `allow_pickle=False`. Why: LangChain's `FAISS.load_local` unpickles, and it trusts the embedder you pass in (lesson G.2).

In [ ]:
# Your code here (Task 6). Blank-file first: write it yourself; no copying from c3_measure.py.

In [ ]:
# CHECK Task 6 (do not edit)
_V6, _c6, _m6 = load_index(INDEX_DIR, RUN)
assert np.array_equal(_V6, V) and _c6 == chunks and _m6 == written, "the good index must load unchanged"


def _refused(d, cfg):
    try:
        load_index(d, cfg)
        return False
    except ValueError:
        return True


_bad = INDEX_ROOT / "bad"
shutil.rmtree(_bad, ignore_errors=True)
shutil.copytree(INDEX_DIR, _bad)
_ls = (_bad / "chunks.jsonl").read_text(encoding="utf-8").split("\n")
(_bad / "chunks.jsonl").write_text("\n".join(_ls[1:]), encoding="utf-8", newline="\n")
assert _refused(_bad, RUN), "a chunks.jsonl that lost a line must be refused"
shutil.rmtree(_bad); shutil.copytree(INDEX_DIR, _bad)
np.save(_bad / "vectors.npy", (V * 2).astype(np.float32))
_mb = json.loads((_bad / "manifest.json").read_text(encoding="utf-8"))
_mb["files"]["vectors.npy"]["sha256"] = hashlib.sha256((_bad / "vectors.npy").read_bytes()).hexdigest()
(_bad / "manifest.json").write_text(json.dumps(_mb, indent=2), encoding="utf-8")
assert _refused(_bad, RUN), "vectors that are not unit length must be refused (even when the manifest hash was updated)"
shutil.rmtree(_bad); shutil.copytree(INDEX_DIR, _bad)
_mb = json.loads((_bad / "manifest.json").read_text(encoding="utf-8"))
_mb["embed_hash"] = "000000000000"
(_bad / "manifest.json").write_text(json.dumps(_mb, indent=2), encoding="utf-8")
assert _refused(_bad, RUN), "a manifest whose embed_hash does not match its embedding block must be refused"
assert _refused(INDEX_DIR, {**RUN, "chunker": {**CFG, "chunk_size": 2000, "chunk_overlap": 400}}), "a run config with another chunker must be refused"
assert _refused(INDEX_DIR, {**RUN, "embedding": {**MANIFEST["embedding"], "model_id": L3, "revision": SPEC_L3["revision"]}}), "a run config with another model must be refused"
assert not _refused(INDEX_DIR, {**RUN, "retrieval": {"k": 8}}), "k is not an index setting: the index must still load"
shutil.rmtree(_bad)
print("Task 6 OK: the good index loads; 5 bad cases are refused; a k change still loads.")

## Given (see C4): the consumer side

This is what C4 will do with your three files: load them through **your** `load_index`, embed the question with the **same** model, score every chunk (exact cosine), and break ties by `chunk_id` (06 §3.1). The D-002 evidence is on p.10.

In [ ]:
V_l, chunks_l, man_l = load_index(INDEX_DIR, RUN)
q = m6.encode([Q_D002], normalize_embeddings=True)[0].astype(np.float32)
scores = V_l @ q
top4 = sorted(range(len(scores)), key=lambda i: (-float(scores[i]), chunks_l[i]["chunk_id"]))[:4]
for r, i in enumerate(top4, 1):
    c = chunks_l[i]
    print(r, c["chunk_id"], "p.", c["page_segments"][0]["page"], round(float(scores[i]), 4), repr(c["text"][:50]))

## Task 7: the rebuild rule in your own words

In 2-3 sentences: when must the index be rebuilt, when can it be reused, and what does the content-keyed cache save you? Use at least one number from Tasks 1-5.

*Template:* "The index must be rebuilt when ____. It can be reused when ____. After the page-10 edit the cache re-embedded only ____ of 97 chunks, because ____."

In [ ]:
my_answer_rebuild_rule = ""   # write your 2-3 sentences inside the quotes

In [ ]:
# CHECK Task 7 (do not edit)
assert len(my_answer_rebuild_rule.strip()) >= 80, "write 2-3 full sentences"
assert any(ch.isdigit() for ch in my_answer_rebuild_rule), "use at least one of your measured numbers"
print("Task 7 recorded. The tutor grades it with the Transfer rubric (05 §4.2).")

## Task 8 (optional, paid API, off by default): the same cache with Cohere

Only if you choose to, and the tutor agrees: set `RUN_COHERE = True`. The cell embeds **8** chunk texts twice with Cohere (as `search_document`, then the same 8 texts as `search_query`) through **your** `cached_embed`, so 16 texts are sent once; a re-run is served from the cache. It never prints your key.

**Expected (from the O2 smoke test only):** `embed-v4.0`, 1,536 dimensions. **UNVERIFIED** until you run it: whether the raw vectors are unit length, and how different the `search_query` and `search_document` vectors of the same text are. Record what you see in the session log.

**Predict first:** will the 8 `search_query` vectors come from the cache after the 8 `search_document` ones were stored? Why?

In [ ]:
RUN_COHERE = False
my_prediction_cohere_query_from_cache = None   # "yes" or "no", BEFORE you set RUN_COHERE = True

In [ ]:
if RUN_COHERE:
    assert my_prediction_cohere_query_from_cache in ("yes", "no"), "write your prediction first"
    from dotenv import load_dotenv
    import cohere
    load_dotenv(WORK / ".env")
    assert os.getenv("COHERE_API_KEY"), "COHERE_API_KEY is missing in .env"
    _model = os.getenv("COHERE_EMBED_MODEL_ID") or "embed-v4.0"
    _co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

    def cohere_encoder(input_type):
        def enc(batch):
            r = _co.embed(texts=list(batch), model=_model, input_type=input_type, embedding_types=["float"])
            return np.asarray(r.embeddings.float_, dtype=np.float32)
        return enc

    _spec_d = dict(provider="cohere", model_id=_model, revision=None, input_type="search_document", preproc_version="none")
    _spec_q = {**_spec_d, "input_type": "search_query"}
    _Vd, _sd = cached_embed(texts[:8], _spec_d, cohere_encoder("search_document"), LAB_CACHE)
    _Vq, _sq = cached_embed(texts[:8], _spec_q, cohere_encoder("search_query"), LAB_CACHE)
    _raw = [np.load(LAB_CACHE / f"{embedding_cache_key(t, **_spec_d)}.npy") for t in texts[:8]]
    print("model", _model, "| dim", _Vd.shape[1], "| documents", _sd, "| queries", _sq)
    print("raw norms (search_document):", [round(float(np.linalg.norm(x)), 4) for x in _raw])
    print("cos(search_query, search_document) for the same text:", [round(float(a @ b), 4) for a, b in zip(_Vq, _Vd)])
else:
    print("Skipped (RUN_COHERE = False). Nothing was sent to Cohere.")

## After the lab

- Log in the session record: which tasks needed a hint (and which level), your predictions vs the measured values, and any failing CHECK you fixed yourself.
- The tutor grades `coding_independence_0_3` on Tasks 1, 3 and 6, and `debugging_0_3` on any failing CHECK you diagnosed and fixed.
- The folders `index/c3_lab/` and `.cache/emb_c3_lab/` are git-ignored; delete them whenever you like.
- Then go back to the lesson, Part J.